# Problema X23B61T0Y13B11 — Desenvolvendo as Integrais da Equação Solução 2D

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x23b61t0y13b11.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição do problema

O Cap. 5 (Seção 5.3) apresenta a **solução geral** do problema X23Y13 — placa retangular com as quatro condições de contorno não homogêneas — como uma soma de quatro integrais de contorno, sem desenvolvê-las. Este notebook desenvolve essas integrais para forçamentos concretos:

- **$x = 0$** (tipo 2, variação **B6**): fluxo oscilatório $q''_0\cos(\omega t)$
- **$x = a$** (tipo 3, B1): convecção com fluido a $T_\infty$ ($f_2 = h_2 T_\infty$)
- **$y = 0$** (tipo 1, B1): temperatura prescrita $T_1$ ($f_1 = T_1$)
- **$y = b$** (tipo 3, B1): convecção com fluido a $T_\infty$ ($f_3 = h_3 T_\infty$)
- **Condição inicial:** $T(x,y,0) = 0$

Na nomenclatura de Beck, o problema é o **X23B61T0Y13B11**. É o primeiro exemplo do livro com variação temporal do tipo B6 (forçamento harmônico) — que dá origem ao fenômeno da **onda térmica**.

## As integrais desenvolvidas

A função de Green é o produto $G_{X23}\cdot G_{Y13}$, com autovalores das duas equações transcendentais

$$\beta_m\tan\beta_m = B_2 = \frac{h_2 a}{k}, \qquad \gamma_n\cot\gamma_n = -B_3 = -\frac{h_3 b}{k}$$

As **integrais espaciais** são elementares:

$$\int_0^b \sin\frac{\gamma_n y'}{b}\,dy' = \frac{b}{\gamma_n}(1-\cos\gamma_n), \qquad \int_0^a \cos\frac{\beta_m x'}{a}\,dx' = \frac{a}{\beta_m}\sin\beta_m$$

As **integrais temporais** têm dois núcleos: para forçamento constante,

$$\int_0^t e^{-D_{mn}\alpha(t-\tau)}\,d\tau = \frac{1-e^{-D_{mn}\alpha t}}{D_{mn}\alpha}$$

e para o forçamento oscilatório (a novidade deste problema),

$$\int_0^t e^{-D_{mn}\alpha(t-\tau)}\cos(\omega\tau)\,d\tau = \frac{D_{mn}\alpha\cos(\omega t) + \omega\sin(\omega t) - D_{mn}\alpha\,e^{-D_{mn}\alpha t}}{(D_{mn}\alpha)^2 + \omega^2}$$

Os dois primeiros termos do numerador são o **regime periódico permanente**; o último é o transiente de acomodação.

## Bibliotecas utilizadas

- **NumPy**: computação numérica (`np.einsum` para as somas duplas das séries).
- **Matplotlib**: gráficos.
- **SciPy**: `brentq` (raízes das equações transcendentais) e `scipy.sparse` (verificação independente por diferenças finitas).

In [ ]:
import numpy as np                        # computação numérica
import matplotlib.pyplot as plt           # gráficos
from scipy.optimize import brentq         # raízes de funções (método de Brent)
import scipy.sparse as sp                 # matrizes esparsas (verificação FD)
import scipy.sparse.linalg as spla        # solver esparso

## Parâmetros do problema

Mesma placa das demais seções do Cap. 5, com fluxo oscilatório de período 20 s.

In [ ]:
a     = 66e-4          # dimensão em x [m]
b     = 66e-4          # dimensão em y [m]
alpha = 1.93e-7        # difusividade térmica [m²/s]
k     = 0.81           # condutividade térmica [W/(m·K)]
q0    = 1e4            # amplitude do fluxo oscilatório em x=0 [W/m²]
omega = 2*np.pi/20     # frequência angular [rad/s] (período de 20 s)
h2    = 25.0           # coeficiente convectivo em x=a [W/(m²·K)]
h3    = 25.0           # coeficiente convectivo em y=b [W/(m²·K)]
Tinf2 = 20.0           # temperatura do fluido em x=a [°C]
Tinf3 = 20.0           # temperatura do fluido em y=b [°C]
T1    = 30.0           # temperatura prescrita em y=0 [°C]

M = 80                 # termos da série na direção x
N = 80                 # termos da série na direção y

B2 = h2*a/k            # Biot modificado da direção x
B3 = h3*b/k            # Biot modificado da direção y
print(f'B2 = {B2:.4f}   B3 = {B3:.4f}')

## Autovalores das duas equações transcendentais

Na direção $x$, cada raiz de $\beta\tan\beta = B_2$ está em $((m-1)\pi,\,(m-1)\pi+\pi/2)$. Na direção $y$, escrevendo $\gamma\cot\gamma = -B_3$ como $g(\gamma) = \gamma\cos\gamma + B_3\sin\gamma = 0$, cada raiz está em $((2n-1)\pi/2,\,n\pi)$.

In [ ]:
def autovalores_x23(B, M):
    """Raízes de beta*tan(beta) = B (caso 4 da tabela de autofunções)."""
    eps = 1e-12
    return np.array([
        brentq(lambda bt: bt*np.tan(bt) - B,
               (m-1)*np.pi + eps, (m-1)*np.pi + np.pi/2 - eps)
        for m in range(1, M+1)
    ])

def autovalores_y13(B, N):
    """Raízes de gama*cot(gama) = -B (caso 7 da tabela)."""
    g = lambda ga: ga*np.cos(ga) + B*np.sin(ga)
    eps = 1e-12
    return np.array([
        brentq(g, (2*n-1)*np.pi/2 + eps, n*np.pi - eps)
        for n in range(1, N+1)
    ])

beta = autovalores_x23(B2, M)
gama = autovalores_y13(B3, N)

# funções de norma (casos 4 e 7 da tabela de autofunções)
Nx = (beta**2 + B2**2) / (beta**2 + B2**2 + B2)
Ny = (gama**2 + B3**2) / (gama**2 + B3**2 + B3)

print('beta_m:', beta[:4])
print('gama_n:', gama[:4])

## Solução com as integrais desenvolvidas

Os quatro termos da equação solução, cada um com seu fator de contorno:

- termo 1 (fluxo em $x'=0$): autofunção avaliada em $x'=0$ ($\cos = 1$), núcleo oscilatório;
- termo 2 (convecção em $x'=a$): fator $\cos\beta_m$, núcleo constante;
- termo 3 (tipo 1 em $y'=0$): derivada da autofunção, fator $\gamma_n/b$, núcleo constante;
- termo 4 (convecção em $y'=b$): fator $\sin\gamma_n$, núcleo constante.

In [ ]:
def solucao_x23y13(x, y, t):
    """T(x,y,t) do Problema X23B61T0Y13B11 (x, y escalares; t vetor)."""
    Bm, Gn = beta[:, None], gama[None, :]
    Dmn = (Bm/a)**2 + (Gn/b)**2                    # decaimento conjunto (M,N)
    Da  = Dmn*alpha

    cosx = np.cos(Bm*x/a)
    siny = np.sin(Gn*y/b)
    base = Nx[:, None]*Ny[None, :]*cosx*siny       # fator modal comum

    int_sin_y = (b/Gn)*(1 - np.cos(Gn))            # integrais espaciais
    int_cos_x = (a/Bm)*np.sin(Bm)

    Et = np.exp(-Da[..., None]*t[None, None, :])   # (M,N,T)

    K_const = (1 - Et)/Da[..., None]               # núcleo constante
    K_osc = ((Da[..., None]*np.cos(omega*t)[None, None, :]
              + omega*np.sin(omega*t)[None, None, :]
              - Da[..., None]*Et) / (Da[..., None]**2 + omega**2))

    c1 = (alpha*q0/k)*(4/(a*b))*base*int_sin_y                    # x'=0
    c2 = (alpha*h2*Tinf2/k)*(4/(a*b))*base*np.cos(Bm)*int_sin_y   # x'=a
    c3 = alpha*T1*(4/(a*b))*base*(Gn/b)*int_cos_x                 # y'=0
    c4 = (alpha*h3*Tinf3/k)*(4/(a*b))*base*np.sin(Gn)*int_cos_x   # y'=b

    return (np.einsum('mn,mnt->t', c1, K_osc)
            + np.einsum('mn,mnt->t', c2, K_const)
            + np.einsum('mn,mnt->t', c3, K_const)
            + np.einsum('mn,mnt->t', c4, K_const))

## Verificação 1: condição inicial

In [ ]:
T_ini = solucao_x23y13(a/2, b/2, np.array([1e-4]))
print(f'T(a/2, b/2, t~0) = {T_ini[0]:.4f} °C   (esperado ~ 0)')

## Verificação 2: regime permanente vs diferenças finitas

Com $\omega = 0$ o fluxo torna-se constante e o problema atinge regime permanente. A equação de Laplace com as quatro condições de contorno é resolvida **independentemente** por diferenças finitas (malha $121\times 121$, fantasmas eliminados nas fronteiras de fluxo/convecção) e comparada com a série para $t \gg$ constante de tempo ($\approx 77$ s). A série usa 300 termos por direção, pois a convergência no contorno é $\mathcal{O}(1/M)$.

In [ ]:
# série com mais termos para o regime permanente
beta_80, gama_80, Nx_80, Ny_80 = beta, gama, Nx, Ny
beta = autovalores_x23(B2, 300)
gama = autovalores_y13(B3, 300)
Nx = (beta**2 + B2**2)/(beta**2 + B2**2 + B2)
Ny = (gama**2 + B3**2)/(gama**2 + B3**2 + B3)
omega_salvo, omega = omega, 0.0
t_perm = np.array([600.0])

# ---- diferenças finitas (regime permanente) ----
n_fd = 121
dx, dy = a/(n_fd-1), b/(n_fd-1)

def indice(i, j):
    return i*n_fd + j          # i: direção x, j: direção y

A_fd = sp.lil_matrix((n_fd*n_fd, n_fd*n_fd))
rhs = np.zeros(n_fd*n_fd)
for i in range(n_fd):
    for j in range(n_fd):
        p = indice(i, j)
        if j == 0:                                    # y=0: T = T1
            A_fd[p, p] = 1.0
            rhs[p] = T1
            continue
        if i == 0:                                    # x=0: fluxo q0
            A_fd[p, indice(1, j)] += 2/dx**2
            rhs[p] -= 2*q0/(k*dx)
        elif i == n_fd-1:                             # x=a: convecção h2
            A_fd[p, indice(n_fd-2, j)] += 2/dx**2
            A_fd[p, p] += -2*h2/(k*dx)
            rhs[p] -= 2*h2*Tinf2/(k*dx)
        else:
            A_fd[p, indice(i-1, j)] += 1/dx**2
            A_fd[p, indice(i+1, j)] += 1/dx**2
        if j == n_fd-1:                               # y=b: convecção h3
            A_fd[p, indice(i, n_fd-2)] += 2/dy**2
            A_fd[p, p] += -2*h3/(k*dy)
            rhs[p] -= 2*h3*Tinf3/(k*dy)
        else:
            A_fd[p, indice(i, j-1)] += 1/dy**2
            A_fd[p, indice(i, j+1)] += 1/dy**2
        A_fd[p, p] += -2/dx**2 - 2/dy**2

T_fd = spla.spsolve(sp.csr_matrix(A_fd), rhs).reshape(n_fd, n_fd)

for (xi, yi) in [(a/2, b/2), (0.0, b/2), (a/2, b/4)]:
    T_serie = solucao_x23y13(xi, yi, t_perm)[0]
    i, j = round(xi/dx), round(yi/dy)
    print(f'({xi/a:.2f}a, {yi/b:.2f}b): série = {T_serie:8.4f} °C | '
          f'FD = {T_fd[i, j]:8.4f} °C | dif = {T_serie - T_fd[i, j]:+.4f} °C')

# restaura os parâmetros do caso oscilatório
omega = omega_salvo
beta, gama, Nx, Ny = beta_80, gama_80, Nx_80, Ny_80

## A onda térmica

Com o forçamento oscilatório, a temperatura em $y = b/2$ mostra o fenômeno da **onda térmica**: a oscilação imposta em $x=0$ propaga-se para o interior com amortecimento exponencial, caracterizado pela profundidade de penetração

$$\delta = \sqrt{\frac{2\alpha}{\omega}} \approx 1{,}1 \text{ mm} \approx 0{,}17\,a$$

Em $x = a/4$ a amplitude já é pequena; além de $x = a/2$, a resposta é praticamente a do valor médio do forçamento.

In [ ]:
cores    = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00']
tracados = ['-', '--', '-.', ':', (0, (5, 1))]

t = np.linspace(0.01, 100.0, 1200)
x_vals  = [0.0, a/4, a/2, 3*a/4, a]
x_names = ['x = 0', 'x = a/4', 'x = a/2', 'x = 3a/4', 'x = a']

fig, ax = plt.subplots()
for x, nome, cor, ls in zip(x_vals, x_names, cores, tracados):
    ax.plot(t, solucao_x23y13(x, b/2, t), label=nome, color=cor,
            linestyle=ls, linewidth=1.3)
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('T(x, b/2, t)  (°C)')
ax.set_title('X23B61T0Y13B11 — onda térmica com fluxo oscilatório em x=0')
ax.legend(ncol=2)
ax.grid(True)
plt.tight_layout()
plt.show()

delta = np.sqrt(2*alpha/omega)
print(f'profundidade de penetração: delta = {delta*1e3:.2f} mm ({delta/a:.2f} a)')

---

## Conclusão

Partindo da solução geral em integrais de contorno da Seção 5.3, o desenvolvimento para forçamentos concretos reduz cada termo a um somatório duplo com núcleos temporais analíticos — inclusive para o forçamento harmônico B6, que introduz o regime periódico permanente e a onda térmica. As duas verificações (condição inicial e comparação com diferenças finitas no regime permanente) confirmam a implementação — mais um exemplo da verificação intrínseca que acompanha as soluções por funções de Green ao longo do livro.